### Ноутбук **`21_per_series_save_expanding_rolling_h4`**

База: [`20_per_series_data_augmentation_vanil.ipynb`](20_per_series_data_augmentation_vanil.ipynb) — те же табличные и **per-series** модели, аугментация (augmentation) коротких рядов, **warnings**.

**Добавлено:**
- Сохранение лучшей **табличной (tabular)** модели на `resist_id` в `MODELS_DIR` (**joblib**) + лист **«Привязка_моделей»** (`model_version_id` = логический `combo_key`; один id может относиться к нескольким `resist_id`). **Per-series:** без pickle — **refit** на фазе 2.
- **Фаза 2:** expanding window (расширяющееся окно) по origin **июнь–ноябрь 2025**, горизонт **4 месяца** как в [`artem_forecast.py`](../artem_forecast.py) (`steps=4`): на каждом origin фиксируется (**commit**) только ближайший месяц; шаги 2–4 перезаписываются на следующем origin (опционально лист **rolling fan**).

**Excel:** `OUT_XLSX` из конфигурации.


In [ ]:
import json
import joblib
import re
import sys
import time
from pathlib import Path

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from artem_forecast import PredictOparationCount

_ARTEM = PredictOparationCount()
from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    GradientBoostingRegressor,
)

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

HAS_PMDARIMA = True
try:
    import pmdarima  # noqa: F401
except ImportError:
    HAS_PMDARIMA = False
    print("pmdarima не установлен — конфигурация AutoARIMA будет пропущена")

HAS_STATSFORECAST = False
try:
    import statsforecast  # noqa: F401

    HAS_STATSFORECAST = True
except Exception as e:
    print(f"statsforecast недоступен — SF-* (Nixtla) будут пропущены ({type(e).__name__})")

HAS_PROPHET = False
try:
    import prophet  # noqa: F401

    HAS_PROPHET = True
except Exception as e:
    print(f"prophet недоступен — Prophet-default будет пропущен ({type(e).__name__})")

HAS_TBATS = False
try:
    import tbats  # noqa: F401

    HAS_TBATS = True
except Exception as e:
    print(f"tbats недоступен — TBATS будет пропущен ({type(e).__name__})")

HAS_NEURALFORECAST = False
try:
    import neuralforecast  # noqa: F401

    HAS_NEURALFORECAST = True
except Exception as e:
    print(f"neuralforecast недоступен — NF-NHITS / NF-NBEATS будут пропущены ({type(e).__name__})")

%matplotlib inline
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message="divide by zero encountered in log",
    category=RuntimeWarning,
)

#### Конфигурация

In [ ]:
RESIST_ID = 124  # не используется, если задан список RESIST_IDS
RESIST_IDS = [124]  # список resist_id; дубликаты убираются автоматически
SUMMARY_TOP_PER_RID = 5  # строк в сводке в ноутбуке (на каждый resist_id)
EXCEL_ITOG_TOP = 20  # строк на лист «Итог» в Excel (совместимость с аналитикой топ-20)

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

# Per-series: StatsForecast (Nixtla) — месячная частота (month start) и сезонность 12.
STATSFORECAST_FREQ = "MS"
MONTHLY_SEASON_LENGTH = 12

# Интермиттирующие модели (Croston, TSB): если доля нулей в train ниже порога — fallback на среднее.
INTERMITTENT_ZERO_MIN_FRACTION = 0.15

# NeuralForecast (NHITS/NBEATS): мало шагов обучения — пилот; увеличение сильно замедляет ноутбук.
NEURALFORECAST_MAX_STEPS = 30

# Набор C: whitelist calculate_parameter (flatten); resist_10 вместо overtime/cross — см. excel_workload_model.md.
CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_employFact_7",
    "calc_resist_10",
]

LAG_FEATURES = ["lag_1", "lag_12", "rolling_mean_3"]
USE_SHORT_SERIES_AUGMENTATION = True
AUGMENTATION_SEED = 42
_AUG_RNG = np.random.default_rng(AUGMENTATION_SEED)

OUT_XLSX = Path("../data/processed/21_per_series_save_expanding_rolling_h4.xlsx")
MODELS_DIR = Path("../data/processed/21_best_models")
ROLLING_HORIZON_STEPS = 4
ORIGIN_MONTHS_2025 = [6, 7, 8, 9, 10, 11]  # последний месяц в train; commit = следующий месяц … декабрь
TABULAR_ROLLING_RECURSIVE_FAN = False  # True: 4 шага рекурсивно (lags от прогноза) для fan табличных моделей


#### Загрузка данных и подготовка (с calc_flat)

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})

df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = pd.to_numeric(df_model39[col], errors="coerce")
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Feature Engineering: lag_1, lag_12, rolling_mean_3; calc_* — lag1–lag6 и lag12

In [ ]:
# Лаги и наборы признаков строятся в ячейке «Табличные + per-series» для каждого resist_id.


#### Проверка пропусков

In [ ]:
# Проверка пропусков выполняется в цикле обучения для каждого df.


#### Наборы признаков (Feature Sets) и split

In [ ]:
# FEATURE_SETS и train/test split задаются в цикле по RESIST_IDS.


#### Метрики + вспомогательные функции

In [ ]:
def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo

#### Табличные модели: 14 конфигураций × 6 наборов признаков = 84 комбинации

In [ ]:
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=PerformanceWarning)

TABULAR_OP_PLACEHOLDER = "— (все операции)"


def _statsmodels_module_tag(warning) -> str:
    fn = (getattr(warning, "filename", None) or "").lower()
    if not fn:
        return ""
    if "holtwinters" in fn:
        return "holtwinters"
    if "kalman_filter" in fn:
        return "kalman_filter"
    if "sarimax" in fn:
        return "sarimax"
    if "statespace" in fn:
        return "statespace"
    if "theta" in fn and "forecasting" in fn:
        return "theta"
    return ""


def interpret_warning(msg: str, category) -> str:
    cname = getattr(category, "__name__", str(category))
    s = str(msg).lower()
    if "maximum likelihood optimization failed" in s or "convergence" in cname.lower():
        return (
            "Оптимизация MLE не сошлась: возможны короткая история, экстремальные значения ряда, "
            "плохая масштабируемость или неудачная спецификация модели."
        )
    if "invalid value encountered in scalar add" in s or "aicc" in s:
        return (
            "Численная нестабильность при расчёте AICc; часто следствие плохого фита или NaN/Inf."
        )
    if "overflow encountered in matmul" in s:
        return (
            "Переполнение при оценке ошибок (Holt-Winters): расходимость или экстремальные уровни ряда."
        )
    if "too few observations" in s and "seasonal" in s:
        return "Недостаточно наблюдений для стартовых параметров сезонной части SARIMAX."
    if "invalid value encountered in divide" in s:
        return "Некорректное деление: ноль или NaN; вырожденная дисперсия / нестабильная модель."
    if "runtime" in cname.lower():
        return "RuntimeWarning: деление на ноль, переполнение или NaN в вычислениях."
    if "sklearn" in (getattr(category, "__module__", "") or ""):
        return "Sklearn: не сходимость итераций или плохо обусловленные признаки."
    return (
        "См. текст сообщения; типично: короткий или нестабильный ряд, экстремальные значения, "
        "численная неустойчивость."
    )


def append_warning_rows(
    rows: list,
    *,
    resist_id,
    operation_id,
    model_kind: str,
    model_label: str,
    phase: str,
    n_train_points,
    caught,
):
    for w in caught:
        msg = str(w.message)
        cat = w.category
        if issubclass(cat, PerformanceWarning):
            continue
        fn = (getattr(w, "filename", None) or "").lower()
        interp = interpret_warning(msg, cat)
        if "invalid value encountered in divide" in str(msg).lower() and "kalman" in fn:
            interp = (
                "Некорректное деление в фильтре Калмана: вырожденная дисперсия / NaN в state space."
            )
        rows.append(
            {
                "resist_id": resist_id,
                "operation_id": operation_id,
                "model_kind": model_kind,
                "model_label": model_label,
                "phase": phase,
                "warning_category": getattr(cat, "__name__", str(cat)),
                "message": msg,
                "interpretation": interp,
                "n_train_points": n_train_points if model_kind == "per_series" else "",
                "statsmodels_module": _statsmodels_module_tag(w),
            }
        )


def train_ets_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    trend="add",
    damped_trend=False,
    seasonal=None,
    seasonal_periods=None,
):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if seasonal is not None and seasonal_periods is not None:
                min_hist = max(4, 2 * int(seasonal_periods))
                if len(y_hist) < min_hist:
                    raise ValueError("too short for seasonal")
            elif len(y_hist) < 4:
                raise ValueError("too short")
            if trend == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul trend requires positive series")
            if seasonal == "mul" and np.any(y_hist <= 0):
                raise ValueError("mul seasonal requires positive series")
            model = ExponentialSmoothing(
                y_hist,
                trend=trend,
                damped_trend=damped_trend,
                seasonal=seasonal,
                seasonal_periods=seasonal_periods if seasonal is not None else None,
            )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                fit = model.fit(optimized=True, use_brute=False)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = fit.forecast(n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_theta_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    theta_deseasonalize=True,
    theta_method="auto",
):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 4:
                raise ValueError("too short")
            if theta_method == "multiplicative" and np.any(y_hist <= 0):
                raise ValueError("theta multiplicative requires positive series")
            tm = ThetaModel(
                y_hist,
                period=12,
                deseasonalize=theta_deseasonalize,
                method=theta_method,
            )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                res = tm.fit()
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = res.forecast(n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_sarimax_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    sarimax_preset="default",
):
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 8:
                raise ValueError("too short")
            if len(y_hist) >= 24:
                if sarimax_preset == "airline":
                    order = (0, 1, 1)
                    seasonal_order = (0, 1, 1, 12)
                else:
                    order = (1, 1, 1)
                    seasonal_order = (1, 1, 1, 12)
            else:
                order = (1, 1, 1)
                seasonal_order = (0, 0, 0, 0)
            model = SARIMAX(
                y_hist,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                fit = model.fit(disp=False)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = fit.forecast(n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_auto_arima_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    auto_arima_kwargs=None,
):
    from pmdarima import auto_arima

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    base_kw = dict(
        seasonal=True,
        m=12,
        suppress_warnings=False,
        error_action="ignore",
        stepwise=True,
        max_p=3,
        max_q=3,
        max_P=2,
        max_Q=2,
    )
    if auto_arima_kwargs:
        base_kw.update(auto_arima_kwargs)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if len(y_hist) < 8:
                raise ValueError("too short")
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                aa = auto_arima(y_hist, **base_kw)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fc = aa.predict(n_periods=n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def prepare_per_series_train(train_part: pd.DataFrame):
    """sort по date → _delete_null_value → при флаге и 0 < len < 24 — _addition_siries_of_random_values.
    Возвращает (y_chrono, ds_DatetimeIndex, n_eff, n_after_trim).
    """
    train_sorted = train_part.sort_values("date")
    idx = pd.to_datetime(train_sorted["date"])
    y = train_sorted["fact_value"].values.astype(float)
    s = pd.Series(y, index=idx)
    s = _ARTEM._delete_null_value(s)
    n_trim = len(s)
    if n_trim == 0:
        return (
            np.array([], dtype=float),
            pd.DatetimeIndex([], dtype="datetime64[ns]"),
            0,
            0,
        )
    if USE_SHORT_SERIES_AUGMENTATION and n_trim < 24:
        s_aug = _ARTEM._addition_siries_of_random_values(s, rng=_AUG_RNG)
        s_aug = s_aug.sort_index()
        return (
            s_aug.values.astype(float),
            pd.DatetimeIndex(s_aug.index),
            len(s_aug),
            n_trim,
        )
    s = s.sort_index()
    return (
        s.values.astype(float),
        pd.DatetimeIndex(s.index),
        len(s),
        n_trim,
    )


def _sf_train_df_from_y_ds(y: np.ndarray, ds: pd.DatetimeIndex) -> pd.DataFrame:
    return pd.DataFrame({"unique_id": "s0", "ds": ds, "y": y.astype(float)})


def _sf_train_df(train_part: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "unique_id": "s0",
            "ds": pd.to_datetime(train_part["date"]),
            "y": train_part["fact_value"].values.astype(float),
        }
    )


def _sf_forecast_column(fc_df: pd.DataFrame) -> np.ndarray:
    meta = {"unique_id", "ds"}
    num_cols = [c for c in fc_df.columns if c not in meta]
    if not num_cols:
        raise ValueError("statsforecast: нет колонки прогноза")
    return fc_df[num_cols[0]].values.astype(float)


def _sf_build_model_list(sf_model_name: str, season_len: int):
    from statsforecast.models import (
        ADIDA,
        AutoARIMA,
        AutoCES,
        AutoETS,
        AutoTheta,
        CrostonSBA,
        DynamicOptimizedTheta,
        HistoricAverage,
        Holt,
        HoltWinters,
        IMAPA,
        MSTL,
        Naive,
        OptimizedTheta,
        RandomWalkWithDrift,
        SeasonalNaive,
        SimpleExponentialSmoothingOptimized,
        TSB,
    )

    m = sf_model_name
    if m == "AutoETS":
        return [AutoETS(season_length=season_len)]
    if m == "AutoTheta":
        return [AutoTheta(season_length=season_len)]
    if m == "AutoCES":
        return [AutoCES(season_length=season_len)]
    if m == "AutoARIMA":
        return [AutoARIMA(season_length=season_len)]
    if m == "Naive":
        return [Naive()]
    if m == "SeasonalNaive":
        return [SeasonalNaive(season_length=season_len)]
    if m == "HistoricAverage":
        return [HistoricAverage()]
    if m == "RandomWalkWithDrift":
        return [RandomWalkWithDrift()]
    if m == "OptimizedTheta":
        return [OptimizedTheta(season_length=season_len)]
    if m == "DynamicOptimizedTheta":
        return [DynamicOptimizedTheta(season_length=season_len)]
    if m == "MSTL":
        return [MSTL(season_length=season_len)]
    if m == "Holt":
        return [Holt()]
    if m == "HoltWinters":
        return [HoltWinters(season_length=season_len)]
    if m == "SimpleExponentialSmoothingOptimized":
        return [SimpleExponentialSmoothingOptimized()]
    if m == "CrostonSBA":
        return [CrostonSBA()]
    if m == "TSB":
        return [TSB(alpha_d=0.3, alpha_p=0.3)]
    if m == "ADIDA":
        return [ADIDA()]
    if m == "IMAPA":
        return [IMAPA()]
    raise ValueError(f"Неизвестная модель StatsForecast: {sf_model_name}")


def train_statsforecast_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    sf_model_name: str,
    season_len: int,
):
    from statsforecast import StatsForecast

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if n_tr < 2:
                raise ValueError("too short")
            sf_df = _sf_train_df_from_y_ds(y_hist, ds_fit)
            models = _sf_build_model_list(sf_model_name, season_len)
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                sf = StatsForecast(models=models, freq=STATSFORECAST_FREQ, n_jobs=1)
                sf.fit(sf_df)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fc_df = sf.predict(h=n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = _sf_forecast_column(fc_df)
            if fc.shape[0] != n_fc:
                fc = fc[:n_fc] if fc.shape[0] > n_fc else np.resize(fc, n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_statsforecast_intermittent_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    sf_model_name: str,
    season_len: int,
):
    from statsforecast import StatsForecast

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        zero_frac = float(np.mean(y_hist == 0)) if n_tr else 0.0
        use_intermittent = zero_frac >= INTERMITTENT_ZERO_MIN_FRACTION
        try:
            if n_tr < 2:
                raise ValueError("too short")
            sf_df = _sf_train_df_from_y_ds(y_hist, ds_fit)
            if use_intermittent:
                models = _sf_build_model_list(sf_model_name, season_len)
            else:
                models = _sf_build_model_list("HistoricAverage", season_len)
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                sf = StatsForecast(models=models, freq=STATSFORECAST_FREQ, n_jobs=1)
                sf.fit(sf_df)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fc_df = sf.predict(h=n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = _sf_forecast_column(fc_df)
            if fc.shape[0] != n_fc:
                fc = fc[:n_fc] if fc.shape[0] > n_fc else np.resize(fc, n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_prophet_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    prophet_params=None,
):
    from prophet import Prophet

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    p_kw = {} if not prophet_params else dict(prophet_params)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if n_tr < 4:
                raise ValueError("too short")
            df_p = pd.DataFrame({"ds": ds_fit, "y": y_hist})
            m = Prophet(**p_kw)
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                m.fit(df_p)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            future = pd.DataFrame({"ds": pd.to_datetime(test_part["date"])})
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fcst = m.predict(future)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = fcst["yhat"].values.astype(float)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_ucm_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    ucm_level="local linear trend",
):
    from statsmodels.tsa.statespace.structural import UnobservedComponents

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if n_tr < 2 * MONTHLY_SEASON_LENGTH:
                raise ValueError("too short for seasonal UCM")
            mod = UnobservedComponents(
                y_hist,
                level=ucm_level,
                seasonal=MONTHLY_SEASON_LENGTH,
            )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                res = mod.fit(disp=False)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = res.forecast(n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_tbats_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    tbats_use_box_cox=None,
    tbats_use_arma_errors=None,
):
    from tbats import TBATS

    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if n_tr < 2 * MONTHLY_SEASON_LENGTH:
                raise ValueError("too short for TBATS")
            tb_kw = {"seasonal_periods": [MONTHLY_SEASON_LENGTH]}
            if tbats_use_box_cox is not None:
                tb_kw["use_box_cox"] = tbats_use_box_cox
            if tbats_use_arma_errors is not None:
                tb_kw["use_arma_errors"] = tbats_use_arma_errors
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                estimator = TBATS(**tb_kw)
                fitted = estimator.fit(y_hist)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_fc:
                warnings.simplefilter("always")
                fc = fitted.forecast(steps=n_fc)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="forecast",
                n_train_points=n_tr,
                caught=w_fc,
            )
            fc = np.asarray(fc, dtype=float).ravel()
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


def train_neuralforecast_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    nf_model_name: str,
    nf_max_steps=None,
):
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NBEATS, NHITS

    max_steps = (
        int(nf_max_steps)
        if nf_max_steps is not None
        else NEURALFORECAST_MAX_STEPS
    )
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist, ds_fit, n_tr, _n_trim_raw = prepare_per_series_train(train_part)
        n_fc = len(test_part)
        try:
            if n_tr < max(8, n_fc + 2):
                raise ValueError("too short for neural forecast")
            sf_df = _sf_train_df_from_y_ds(y_hist, ds_fit)
            inp = min(24, max(2, n_tr - n_fc - 1))
            inp = max(inp, 2)
            if nf_model_name == "NHITS":
                model = NHITS(
                    h=n_fc,
                    input_size=inp,
                    max_steps=max_steps,
                    accelerator="cpu",
                    devices=1,
                )
            else:
                model = NBEATS(
                    h=n_fc,
                    input_size=inp,
                    max_steps=max_steps,
                    accelerator="cpu",
                    devices=1,
                )
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                nf = NeuralForecast(models=[model], freq=STATSFORECAST_FREQ)
                nf.fit(df=sf_df)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                fc_df = nf.predict()
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="predict",
                n_train_points=n_tr,
                caught=w_pred,
            )
            fc = _sf_forecast_column(fc_df)
            if fc.shape[0] != n_fc:
                fc = fc[:n_fc] if fc.shape[0] > n_fc else np.resize(fc, n_fc)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


# Расширенный набор per-series: время прогона (runtime) и число warnings существенно растут.
SERIES_CONFIGS = {
    "ETS-add": {"fn": "ets", "trend": "add", "damped_trend": False, "seasonal": None, "seasonal_periods": None},
    "ETS-damped": {"fn": "ets", "trend": "add", "damped_trend": True, "seasonal": None, "seasonal_periods": None},
    "ETS-seasonal-add": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": False,
        "seasonal": "add",
        "seasonal_periods": 12,
    },
    "ETS-seasonal-damped": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": True,
        "seasonal": "add",
        "seasonal_periods": 12,
    },
    "ETS-mul-trend": {"fn": "ets", "trend": "mul", "damped_trend": False, "seasonal": None, "seasonal_periods": None},
    "ETS-seasonal-mul": {
        "fn": "ets",
        "trend": "add",
        "damped_trend": False,
        "seasonal": "mul",
        "seasonal_periods": 12,
    },
    "Theta-default": {"fn": "theta"},
    "Theta-no-deseason": {"fn": "theta", "theta_deseasonalize": False},
    "Theta-mult-method": {"fn": "theta", "theta_method": "multiplicative"},
    "SARIMAX-default": {"fn": "sarimax"},
    "SARIMAX-airline": {"fn": "sarimax", "sarimax_preset": "airline"},
}
if HAS_PMDARIMA:
    SERIES_CONFIGS["AutoARIMA"] = {"fn": "auto_arima"}
    SERIES_CONFIGS["AutoARIMA-nonseasonal"] = {
        "fn": "auto_arima",
        "auto_arima_kwargs": {"seasonal": False, "m": 1},
    }
    SERIES_CONFIGS["AutoARIMA-BIC"] = {
        "fn": "auto_arima",
        "auto_arima_kwargs": {"information_criterion": "bic"},
    }

if HAS_STATSFORECAST:
    _sl = MONTHLY_SEASON_LENGTH
    SERIES_CONFIGS.update(
        {
            "SF-AutoETS": {"fn": "statsforecast", "sf_model": "AutoETS", "season_len": _sl},
            "SF-AutoTheta": {"fn": "statsforecast", "sf_model": "AutoTheta", "season_len": _sl},
            "SF-AutoCES": {"fn": "statsforecast", "sf_model": "AutoCES", "season_len": _sl},
            "SF-AutoARIMA": {"fn": "statsforecast", "sf_model": "AutoARIMA", "season_len": _sl},
            "SF-Naive": {"fn": "statsforecast", "sf_model": "Naive", "season_len": _sl},
            "SF-SeasonalNaive": {"fn": "statsforecast", "sf_model": "SeasonalNaive", "season_len": _sl},
            "SF-HistoricAverage": {"fn": "statsforecast", "sf_model": "HistoricAverage", "season_len": _sl},
            "SF-RandomWalkWithDrift": {"fn": "statsforecast", "sf_model": "RandomWalkWithDrift", "season_len": _sl},
            "SF-OptimizedTheta": {"fn": "statsforecast", "sf_model": "OptimizedTheta", "season_len": _sl},
            "SF-DynamicOptimizedTheta": {"fn": "statsforecast", "sf_model": "DynamicOptimizedTheta", "season_len": _sl},
            "SF-MSTL": {"fn": "statsforecast", "sf_model": "MSTL", "season_len": _sl},
            "SF-Holt": {"fn": "statsforecast", "sf_model": "Holt", "season_len": _sl},
            "SF-HoltWinters": {"fn": "statsforecast", "sf_model": "HoltWinters", "season_len": _sl},
            "SF-SES-Optimized": {
                "fn": "statsforecast",
                "sf_model": "SimpleExponentialSmoothingOptimized",
                "season_len": _sl,
            },
            "SF-CrostonSBA": {"fn": "statsforecast_intermittent", "sf_model": "CrostonSBA", "season_len": _sl},
            "SF-TSB": {"fn": "statsforecast_intermittent", "sf_model": "TSB", "season_len": _sl},
            "SF-ADIDA": {"fn": "statsforecast_intermittent", "sf_model": "ADIDA", "season_len": _sl},
            "SF-IMAPA": {"fn": "statsforecast_intermittent", "sf_model": "IMAPA", "season_len": _sl},
        }
    )

if HAS_PROPHET:
    SERIES_CONFIGS["Prophet-default"] = {"fn": "prophet", "prophet_params": {}}
    SERIES_CONFIGS["Prophet-yearly"] = {
        "fn": "prophet",
        "prophet_params": {
            "yearly_seasonality": True,
            "weekly_seasonality": False,
            "daily_seasonality": False,
        },
    }
    SERIES_CONFIGS["Prophet-tuned"] = {
        "fn": "prophet",
        "prophet_params": {
            "seasonality_mode": "multiplicative",
            "changepoint_prior_scale": 0.05,
            "n_changepoints": 5,
        },
    }

SERIES_CONFIGS["UCM-local-linear-seasonal"] = {"fn": "ucm"}
SERIES_CONFIGS["UCM-local-level-seasonal"] = {
    "fn": "ucm",
    "ucm_level": "local level",
}

if HAS_TBATS:
    SERIES_CONFIGS["TBATS"] = {"fn": "tbats"}
    SERIES_CONFIGS["TBATS-no-boxcox"] = {"fn": "tbats", "tbats_use_box_cox": False}
    SERIES_CONFIGS["TBATS-no-arma"] = {"fn": "tbats", "tbats_use_arma_errors": False}

if HAS_NEURALFORECAST:
    SERIES_CONFIGS["NF-NHITS"] = {"fn": "neuralforecast", "nf_model": "NHITS"}
    SERIES_CONFIGS["NF-NBEATS"] = {"fn": "neuralforecast", "nf_model": "NBEATS"}
    SERIES_CONFIGS["NF-NHITS-moresteps"] = {
        "fn": "neuralforecast",
        "nf_model": "NHITS",
        "nf_max_steps": 45,
    }
    SERIES_CONFIGS["NF-NBEATS-fewersteps"] = {
        "fn": "neuralforecast",
        "nf_model": "NBEATS",
        "nf_max_steps": 15,
    }


def _build_model_configs() -> dict:
    """Фабрика: каждый вызов создаёт свежие (не обученные) экземпляры моделей."""
    return {
        "LinearRegression": LinearRegression(),
        "DT-shallow": DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=42),
        "DT-deep": DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=42),
        "DT-mid": DecisionTreeRegressor(max_depth=8, min_samples_leaf=4, random_state=42),
        "DT-conservative": DecisionTreeRegressor(
            max_depth=5, min_samples_leaf=10, min_samples_split=15, random_state=42,
        ),
        "DT-ccp": DecisionTreeRegressor(
            max_depth=15, min_samples_leaf=2, ccp_alpha=0.02, random_state=42,
        ),
        "LGBM-default": LGBMRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbose=-1, random_state=42,
        ),
        "LGBM-tuned": LGBMRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            num_leaves=63, min_child_samples=10, verbose=-1, random_state=42,
        ),
        "XGB-default": XGBRegressor(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            verbosity=0, random_state=42,
        ),
        "XGB-regularized": XGBRegressor(
            n_estimators=800, max_depth=8, learning_rate=0.05,
            reg_alpha=0.1, reg_lambda=5, verbosity=0, random_state=42,
        ),
        "RF-default": RandomForestRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1,
        ),
        "ET-default": ExtraTreesRegressor(
            n_estimators=300, max_depth=12, min_samples_leaf=2,
            random_state=42, n_jobs=-1,
        ),
        "HistGBM-default": HistGradientBoostingRegressor(
            max_iter=500, max_depth=6, learning_rate=0.1,
            random_state=42,
        ),
        "GBR-default": GradientBoostingRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=42,
        ),
    }

import sys
from pathlib import Path

try:
    _ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from scripts.calc_lag_features import (
    add_calc_shift_lags,
    calc_lag_columns_for_sets,
    DEFAULT_CALC_SHIFTS,
)

all_results: list = []
all_predictions: dict = {}
df_by_rid: dict = {}
mask_test_by_rid: dict = {}
summary_parts: list[str] = []
warning_rows: list[dict] = []

feature_sets_by_rid = {}

for rid in RESIST_IDS:
    df = df_model39[df_model39["resist_id"] == rid].copy()
    df["op_code"] = df["operation_id"].astype("category").cat.codes

    group = df.groupby("operation_id")
    df["lag_1"] = group["fact_value"].shift(1)
    df["lag_12"] = group["fact_value"].shift(12)
    df["rolling_mean_3"] = group["fact_value"].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].fillna(0)
    df = df.infer_objects(copy=False)

    miss = df.isnull().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        df = df.fillna(0)

    df, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(df, drop_scalar_calc=True)
    b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)

    BASE_CORE = ["op_code", "month", "year", "workDayMonth_2"]
    BASE_WITH_MINUTE = BASE_CORE + ["calculate_minute"]
    FEATURE_SETS = {
        "A": BASE_CORE,
        "B": BASE_WITH_MINUTE + b_flat,
        "C": BASE_WITH_MINUTE + c_flat,
        "A+lags": BASE_CORE + LAG_FEATURES,
        "B+lags": BASE_WITH_MINUTE + b_flat + LAG_FEATURES,
        "C+lags": BASE_WITH_MINUTE + c_flat + LAG_FEATURES,
    }

    mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
    mask_train = ~mask_test
    feature_sets_by_rid[rid] = {k: list(v) for k, v in FEATURE_SETS.items()}
    y_train = df.loc[mask_train, "fact_value"]
    y_test = df.loc[mask_test, "fact_value"]

    df_by_rid[rid] = df
    mask_test_by_rid[rid] = mask_test

    results_rid: list = []
    preds_rid: dict = {}

    n_test = int(mask_test.sum())
    t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)

    if mask_train.sum() == 0:
        y_zero = np.zeros(n_test)
        t_z, s_z, q_z = calc_metrics(y_test.values, y_zero)

        model_configs_dummy = _build_model_configs()
        for fs_name, fs_cols in FEATURE_SETS.items():
            for config_name in model_configs_dummy:
                combo_key = f"{config_name} | {fs_name}"
                results_rid.append({
                    "resist_id": rid,
                    "Конфигурация": config_name,
                    "Набор признаков": fs_name,
                    "Тип": "табличная",
                    "Точность, %": t_z,
                    "Стабильность, %": s_z,
                    "Качество, %": q_z,
                    "Время, сек": 0.0,
                    "Точн. fact_predict, %": t_fp,
                    "Стаб. fact_predict, %": s_fp,
                    "Кач. fact_predict, %": q_fp,
                })
                preds_rid[combo_key] = y_zero

        for config_name in SERIES_CONFIGS:
            results_rid.append({
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": "—",
                "Тип": "per-series",
                "Точность, %": t_z,
                "Стабильность, %": s_z,
                "Качество, %": q_z,
                "Время, сек": 0.0,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            })
            preds_rid[config_name] = y_zero

        all_results.extend(results_rid)
        all_predictions[rid] = preds_rid
        q_best = max(r["Качество, %"] for r in results_rid)
        summary_parts.append(
            f"{rid} baseline:{q_fp:.0f}%  forecast:{q_best:.0f}%"
        )
        print(", ".join(summary_parts))
        continue

    combo_num = 0
    for fs_name, fs_cols in FEATURE_SETS.items():
        model_configs = _build_model_configs()
        X_train_fs = df.loc[mask_train, fs_cols]
        X_test_fs = df.loc[mask_test, fs_cols]

        for config_name, model in model_configs.items():
            combo_num += 1
            combo_key = f"{config_name} | {fs_name}"
            t0 = time.perf_counter()
            n_tab_train = int(mask_train.sum())
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                model.fit(X_train_fs, y_train)
            append_warning_rows(
                warning_rows,
                resist_id=rid,
                operation_id=TABULAR_OP_PLACEHOLDER,
                model_kind="tabular",
                model_label=combo_key,
                phase="fit",
                n_train_points=n_tab_train,
                caught=w_fit,
            )
            with warnings.catch_warnings(record=True) as w_pred:
                warnings.simplefilter("always")
                y_pred = np.maximum(model.predict(X_test_fs), 0)
            append_warning_rows(
                warning_rows,
                resist_id=rid,
                operation_id=TABULAR_OP_PLACEHOLDER,
                model_kind="tabular",
                model_label=combo_key,
                phase="predict",
                n_train_points=n_tab_train,
                caught=w_pred,
            )
            elapsed = round(time.perf_counter() - t0, 2)
            tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
            results_rid.append({
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": fs_name,
                "Тип": "табличная",
                "Точность, %": tochnost,
                "Стабильность, %": stab,
                "Качество, %": kachestvo,
                "Время, сек": elapsed,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            })
            preds_rid[combo_key] = y_pred

    for config_name, cfg in SERIES_CONFIGS.items():
        t0 = time.perf_counter()
        if cfg["fn"] == "ets":
            y_pred = train_ets_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                trend=cfg["trend"],
                damped_trend=cfg["damped_trend"],
                seasonal=cfg.get("seasonal"),
                seasonal_periods=cfg.get("seasonal_periods"),
            )
        elif cfg["fn"] == "theta":
            y_pred = train_theta_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                theta_deseasonalize=cfg.get("theta_deseasonalize", True),
                theta_method=cfg.get("theta_method", "auto"),
            )
        elif cfg["fn"] == "sarimax":
            y_pred = train_sarimax_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                sarimax_preset=cfg.get("sarimax_preset", "default"),
            )
        elif cfg["fn"] == "auto_arima":
            y_pred = train_auto_arima_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                auto_arima_kwargs=cfg.get("auto_arima_kwargs"),
            )
        elif cfg["fn"] == "statsforecast":
            y_pred = train_statsforecast_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                sf_model_name=cfg["sf_model"],
                season_len=cfg["season_len"],
            )
        elif cfg["fn"] == "statsforecast_intermittent":
            y_pred = train_statsforecast_intermittent_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                sf_model_name=cfg["sf_model"],
                season_len=cfg["season_len"],
            )
        elif cfg["fn"] == "prophet":
            y_pred = train_prophet_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                prophet_params=cfg.get("prophet_params"),
            )
        elif cfg["fn"] == "ucm":
            y_pred = train_ucm_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                ucm_level=cfg.get("ucm_level", "local linear trend"),
            )
        elif cfg["fn"] == "tbats":
            y_pred = train_tbats_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                tbats_use_box_cox=cfg.get("tbats_use_box_cox"),
                tbats_use_arma_errors=cfg.get("tbats_use_arma_errors"),
            )
        elif cfg["fn"] == "neuralforecast":
            y_pred = train_neuralforecast_per_series(
                df,
                mask_train,
                mask_test,
                rid,
                config_name,
                warning_rows,
                nf_model_name=cfg["nf_model"],
                nf_max_steps=cfg.get("nf_max_steps"),
            )
        else:
            continue

        elapsed = round(time.perf_counter() - t0, 2)
        tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
        results_rid.append({
            "resist_id": rid,
            "Конфигурация": config_name,
            "Набор признаков": "—",
            "Тип": "per-series",
            "Точность, %": tochnost,
            "Стабильность, %": stab,
            "Качество, %": kachestvo,
            "Время, сек": elapsed,
            "Точн. fact_predict, %": t_fp,
            "Стаб. fact_predict, %": s_fp,
            "Кач. fact_predict, %": q_fp,
        })
        preds_rid[config_name] = y_pred

    all_results.extend(results_rid)
    all_predictions[rid] = preds_rid
    q_best = max(r["Качество, %"] for r in results_rid)
    summary_parts.append(
        f"{rid} baseline:{q_fp:.0f}%  forecast:{q_best:.0f}%"
    )
    print(", ".join(summary_parts))

results = all_results


#### Per-series (расширено в 19): **ETS** (в т.ч. seasonal mul), **Theta** (без десезонализации, multiplicative method), **SARIMAX** (default / airline), **AutoARIMA** (+ nonseasonal, BIC), **StatsForecast** (+ ADIDA, IMAPA с веткой intermittent), **Prophet** (default / yearly-only / tuned), **UCM** (local linear trend / local level), **TBATS** (+ без Box–Cox, без ARMA ошибок), **NeuralForecast** NHITS/NBEATS (+ разные `max_steps`). См. `SERIES_CONFIGS` и `HAS_*`.

In [ ]:
# Определения train_* и SERIES_CONFIGS, а также цикл обучения — в предыдущей ячейке.
pass


#### Сводная таблица результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results["Комбинация"] = df_results.apply(
    lambda r: f"{r['Конфигурация']} | {r['Набор признаков']}" if r["Набор признаков"] != "—" else r["Конфигурация"],
    axis=1,
)
df_results_full = df_results.sort_values(
    ["resist_id", "Качество, %"], ascending=[True, False]
).reset_index(drop=True)
df_results = (
    df_results_full.groupby("resist_id", group_keys=False)
    .head(SUMMARY_TOP_PER_RID)
    .reset_index(drop=True)
)
print(f"Строк в полной сводке: {len(df_results_full)}; в таблице ниже — топ-{SUMMARY_TOP_PER_RID} на каждый resist_id")
df_results


In [ ]:
TOP_N = SUMMARY_TOP_PER_RID
for rid in RESIST_IDS:
    sub = df_results_full[df_results_full["resist_id"] == rid].head(TOP_N)
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, TOP_N * 0.45)))
    metrics_to_plot = ["Точность, %", "Стабильность, %", "Качество, %"]
    for ax, metric in zip(axes, metrics_to_plot):
        df_plot = sub.sort_values(metric, ascending=True)
        colors = ["#2196F3" if t == "табличная" else "#FF9800" for t in df_plot["Тип"]]
        ax.barh(df_plot["Комбинация"], df_plot[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(metric)
    plt.suptitle(f"TOP-{TOP_N} комбинаций (resist_id={rid})", fontsize=14)
    plt.tight_layout()
    plt.show()

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid]
    print(f"\n--- resist_id={rid}: лучший набор по каждой модели ---")
    for cfg_name in df_r["Конфигурация"].unique():
        best_row = df_r[df_r["Конфигурация"] == cfg_name].iloc[0]
        print(f"  {cfg_name:<22} → {best_row['Набор признаков']:<7} | Качество={best_row['Качество, %']:.2f}%")


#### Лучшая конфигурация: детали по операциям

In [ ]:
per_rid_best = {}

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid].sort_values(
        "Качество, %", ascending=False
    )
    if df_r.empty:
        continue
    best_row = df_r.iloc[0]
    best_combo = best_row["Комбинация"]
    best_config = best_row["Конфигурация"]
    best_fs = best_row["Набор признаков"]
    best_type = best_row["Тип"]
    print(f"\n{'='*60}\nresist_id={rid}\nЛучшая комбинация по Качеству: {best_combo} ({best_type})")
    print(f"  Модель: {best_config}, Набор: {best_fs}")
    print(
        f"  Точность={best_row['Точность, %']}% | Стабильность={best_row['Стабильность, %']}% | "
        f"Качество={best_row['Качество, %']}%"
    )

    best_pred = all_predictions[rid][best_combo]
    df = df_by_rid[rid]
    mask_test = mask_test_by_rid[rid]
    df_test = df.loc[mask_test].copy()
    df_test["predict_new"] = best_pred

    rows_op = []
    for (r_id, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
        t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
        t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_new"].values)
        rows_op.append({
            "resist_id": r_id,
            "operation_id": op_id,
            "Точность (fact_predict)": t1,
            "Стабильность (fact_predict)": s1,
            "Качество (fact_predict)": q1,
            f"Точность ({best_combo})": t2,
            f"Стабильность ({best_combo})": s2,
            f"Качество ({best_combo})": q2,
        })

    df_metrics_op = pd.DataFrame(rows_op)
    df_detail = df_test[["resist_id", "operation_id", "year", "month",
                         "fact_value", "fact_predict", "predict_new"]].copy()
    df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
    df_detail = df_detail.rename(columns={
        "fact_value": "fact_value (факт)",
        "fact_predict": "fact_predict (текущий)",
        "predict_new": f"{best_combo} (новый)",
    })
    model_version_id = best_combo
    per_rid_best[rid] = {
        "best_combo": best_combo,
        "best_config": best_config,
        "best_fs": best_fs,
        "best_type": best_type,
        "model_version_id": model_version_id,
        "df_metrics_op": df_metrics_op,
        "df_detail": df_detail,
    }
    print(f"Операций в тесте: {len(df_metrics_op)}")

if RESIST_IDS:
    r0 = RESIST_IDS[0]
    best_combo = per_rid_best[r0]["best_combo"]
    df_metrics_op = per_rid_best[r0]["df_metrics_op"]
    print("\n(первый id в списке: пример df_metrics_op.head())")
    print(df_metrics_op.head(10))


#### Сохранение лучших табличных моделей и фаза 2 (expanding window + rolling h=4)

Выполнять **после** ячейки с `per_rid_best`. Пишет `df_model_binding`, `df_phase2_committed`, `df_rolling_fan`, файлы в `MODELS_DIR`.


In [ ]:
# Сохранение лучших табличных моделей + фаза 2 (expanding window + rolling h=4)
from scripts.nb21_df_feature_sets import build_df_and_feature_sets

MODELS_DIR.mkdir(parents=True, exist_ok=True)


def _ym_pairs_after_origin(origin_y: int, origin_m: int, n: int) -> set[tuple[int, int]]:
    ts = pd.Timestamp(year=origin_y, month=origin_m, day=1) + pd.offsets.MonthBegin(1)
    out = []
    for _ in range(n):
        out.append((int(ts.year), int(ts.month)))
        ts = ts + pd.offsets.MonthBegin(1)
    return set(out)


def _mask_train_until(df: pd.DataFrame, test_year: int, origin_month: int) -> pd.Series:
    return (df["year"] < test_year) | ((df["year"] == test_year) & (df["month"] <= origin_month))


def _mask_test_ym_in(df: pd.DataFrame, pairs: set[tuple[int, int]]) -> pd.Series:
    ym = list(zip(df["year"].astype(int), df["month"].astype(int)))
    return pd.Series([t in pairs for t in ym], index=df.index)


def _slug(s: str, max_len: int = 100) -> str:
    x = re.sub(r"[^a-zA-Z0-9_.-]+", "_", s).strip("_")
    return x[:max_len] if x else "model"


def _dispatch_per_series(
    df,
    mask_train,
    mask_test,
    rid,
    config_name,
    cfg,
    warn_rows,
):
    fn = cfg.get("fn")
    if fn == "ets":
        return train_ets_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            trend=cfg["trend"],
            damped_trend=cfg["damped_trend"],
            seasonal=cfg.get("seasonal"),
            seasonal_periods=cfg.get("seasonal_periods"),
        )
    if fn == "theta":
        return train_theta_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            theta_deseasonalize=cfg.get("theta_deseasonalize", True),
            theta_method=cfg.get("theta_method", "auto"),
        )
    if fn == "sarimax":
        return train_sarimax_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            sarimax_preset=cfg.get("sarimax_preset", "default"),
        )
    if fn == "auto_arima":
        return train_auto_arima_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            auto_arima_kwargs=cfg.get("auto_arima_kwargs"),
        )
    if fn == "statsforecast":
        return train_statsforecast_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            sf_model_name=cfg["sf_model"],
            season_len=cfg["season_len"],
        )
    if fn == "statsforecast_intermittent":
        return train_statsforecast_intermittent_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            sf_model_name=cfg["sf_model"],
            season_len=cfg["season_len"],
        )
    if fn == "prophet":
        return train_prophet_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            prophet_params=cfg.get("prophet_params"),
        )
    if fn == "ucm":
        return train_ucm_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            ucm_level=cfg.get("ucm_level", "local linear trend"),
        )
    if fn == "tbats":
        return train_tbats_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            tbats_use_box_cox=cfg.get("tbats_use_box_cox"),
            tbats_use_arma_errors=cfg.get("tbats_use_arma_errors"),
        )
    if fn == "neuralforecast":
        return train_neuralforecast_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warn_rows,
            nf_model_name=cfg["nf_model"],
            nf_max_steps=cfg.get("nf_max_steps"),
        )
    return None


binding_rows: list[dict] = []
phase2_warn = []
committed_rows: list[dict] = []
fan_rows: list[dict] = []

for rid in RESIST_IDS:
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    mv_id = b["model_version_id"]
    combo = b["best_combo"]
    slug = _slug(f"r{rid}_{combo}")

    if b["best_type"] == "табличная":
        fs_name = b["best_fs"]
        cfg_name = b["best_config"]
        fs_cols = feature_sets_by_rid[rid][fs_name]
        df = df_by_rid[rid]
        m_tr_full = ~((df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS)))
        if m_tr_full.sum() == 0:
            art_path = ""
        else:
            model = _build_model_configs()[cfg_name]
            model.fit(df.loc[m_tr_full, fs_cols], df.loc[m_tr_full, "fact_value"])
            bundle = {
                "model": model,
                "fs_cols": fs_cols,
                "resist_id": rid,
                "combo_key": combo,
                "config_name": cfg_name,
                "fs_name": fs_name,
            }
            art_path = str(MODELS_DIR / f"tabular_{slug}.joblib")
            joblib.dump(bundle, art_path)
        binding_rows.append(
            {
                "resist_id": rid,
                "model_version_id": mv_id,
                "Тип": b["best_type"],
                "Конфигурация": b["best_config"],
                "Набор признаков": fs_name,
                "combo_key": combo,
                "artifact_path": art_path,
                "примечание": "tabular joblib после фазы 1 (train до июня 2025)",
            }
        )
    else:
        binding_rows.append(
            {
                "resist_id": rid,
                "model_version_id": mv_id,
                "Тип": b["best_type"],
                "Конфигурация": b["best_config"],
                "Набор признаков": "—",
                "combo_key": combo,
                "artifact_path": "",
                "примечание": "per-series: без pickle, refit на фазе 2",
            }
        )

df_model_binding = pd.DataFrame(binding_rows)

# --- Фаза 2: по каждому origin — retrain, горизонт 4, commit первого месяца ---
for rid in RESIST_IDS:
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    df, FEATURE_SETS = build_df_and_feature_sets(
        df_model39,
        rid,
        calc_whitelist=CALC_WHITELIST,
        lag_features=LAG_FEATURES,
    )

    for origin_m in ORIGIN_MONTHS_2025:
        commit_m = origin_m + 1
        if commit_m > 12:
            continue
        m_tr = _mask_train_until(df, TEST_YEAR, origin_m)
        pairs = _ym_pairs_after_origin(TEST_YEAR, origin_m, ROLLING_HORIZON_STEPS)
        m_te = _mask_test_ym_in(df, pairs)
        if m_tr.sum() == 0 or not m_te.any():
            continue

        if b["best_type"] == "табличная":
            fs_cols = FEATURE_SETS[b["best_fs"]]
            cfg_name = b["best_config"]
            model = _build_model_configs()[cfg_name]
            model.fit(df.loc[m_tr, fs_cols], df.loc[m_tr, "fact_value"])
            sub_fan = df.loc[m_te].sort_values(["operation_id", "year", "month"]).copy()
            if sub_fan.empty:
                continue
            preds_fan = np.maximum(model.predict(sub_fan[fs_cols]).astype(float), 0.0)
            sub_fan["pred_fan"] = preds_fan
            order_map = {ym: i for i, ym in enumerate(sorted(pairs, key=lambda x: (x[0], x[1])))}
            sub_fan["horizon_step"] = sub_fan.apply(
                lambda r: order_map.get((int(r["year"]), int(r["month"])), -1), axis=1
            )
            sub_c = sub_fan[
                (sub_fan["year"] == TEST_YEAR) & (sub_fan["month"] == commit_m)
            ].copy()
            for _, r in sub_c.iterrows():
                committed_rows.append(
                    {
                        "resist_id": rid,
                        "origin_last_train_month": origin_m,
                        "target_year": int(r["year"]),
                        "target_month": int(r["month"]),
                        "operation_id": r["operation_id"],
                        "fact_value": r["fact_value"],
                        "predict_committed": r["pred_fan"],
                        "best_combo": b["best_combo"],
                        "Тип": b["best_type"],
                    }
                )
            for _, r in sub_fan.iterrows():
                fan_rows.append(
                    {
                        "resist_id": rid,
                        "origin_last_train_month": origin_m,
                        "target_year": int(r["year"]),
                        "target_month": int(r["month"]),
                        "horizon_step": int(r["horizon_step"]),
                        "operation_id": r["operation_id"],
                        "pred_rolling_fan": r["pred_fan"],
                        "best_combo": b["best_combo"],
                    }
                )
        else:
            cfg = SERIES_CONFIGS.get(b["best_config"])
            if cfg is None:
                continue
            pred_vec = _dispatch_per_series(
                df, m_tr, m_te, rid, b["best_config"], cfg, phase2_warn
            )
            if pred_vec is None:
                continue
            idx_te = df.loc[m_te].index
            pv = np.maximum(np.asarray(pred_vec, dtype=float).ravel(), 0.0)
            n = min(len(pv), len(idx_te))
            pred_by_idx = pd.Series(pv[:n], index=idx_te[:n])
            cols = ["resist_id", "operation_id", "year", "month", "fact_value"]
            sub = df.loc[pred_by_idx.index, cols].copy()
            sub["pred_fan"] = pred_by_idx.values
            order_map = {ym: i for i, ym in enumerate(sorted(pairs, key=lambda x: (x[0], x[1])))}
            sub["horizon_step"] = sub.apply(
                lambda r: order_map.get((int(r["year"]), int(r["month"])), -1), axis=1
            )
            sub_c = sub[(sub["year"] == TEST_YEAR) & (sub["month"] == commit_m)].copy()
            for _, r in sub_c.iterrows():
                committed_rows.append(
                    {
                        "resist_id": rid,
                        "origin_last_train_month": origin_m,
                        "target_year": int(r["year"]),
                        "target_month": int(r["month"]),
                        "operation_id": r["operation_id"],
                        "fact_value": r["fact_value"],
                        "predict_committed": r["pred_fan"],
                        "best_combo": b["best_combo"],
                        "Тип": b["best_type"],
                    }
                )
            for _, r in sub.iterrows():
                fan_rows.append(
                    {
                        "resist_id": rid,
                        "origin_last_train_month": origin_m,
                        "target_year": int(r["year"]),
                        "target_month": int(r["month"]),
                        "horizon_step": int(r["horizon_step"]),
                        "operation_id": r["operation_id"],
                        "pred_rolling_fan": r["pred_fan"],
                        "best_combo": b["best_combo"],
                    }
                )

df_phase2_committed = pd.DataFrame(committed_rows)
df_rolling_fan = pd.DataFrame(fan_rows)

if not df_phase2_committed.empty:
    df_p2 = df_phase2_committed[
        (df_phase2_committed["target_year"] == TEST_YEAR)
        & (df_phase2_committed["target_month"].isin(TEST_MONTHS))
    ]
    t2, s2, q2 = calc_metrics(
        df_p2["fact_value"].values, df_p2["predict_committed"].values
    )
    print(f"Фаза 2 (committed Jul–Dec): Точность={t2}% Стабильность={s2}% Качество={q2}% (N={len(df_p2)})")
else:
    print("Фаза 2: нет строк committed (проверьте данные / маски).")

print(f"Привязка моделей: {len(df_model_binding)} строк; артефакты в {MODELS_DIR.resolve()}")


#### Экспорт в Excel

In [ ]:
from openpyxl import load_workbook

out_path = OUT_XLSX
out_path.parent.mkdir(parents=True, exist_ok=True)

WARN_COLS = [
    "resist_id",
    "operation_id",
    "model_kind",
    "model_label",
    "phase",
    "warning_category",
    "message",
    "interpretation",
    "n_train_points",
    "statsmodels_module",
]

sheet_triples = []
for rid in RESIST_IDS:
    df_itog = df_results_full[df_results_full["resist_id"] == rid].head(EXCEL_ITOG_TOP)
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    df_warn = pd.DataFrame([r for r in warning_rows if r["resist_id"] == rid])
    if df_warn.empty:
        df_warn = pd.DataFrame(columns=WARN_COLS)
    else:
        df_warn = df_warn.reindex(columns=WARN_COLS)
    sheet_triples.append((rid, df_itog, b["df_detail"], b["df_metrics_op"], df_warn))

# Листы фазы 2 / привязка (заполняются ячейкой «Сохранение моделей и фаза 2», если выполнена)
df_model_binding = globals().get("df_model_binding")
if df_model_binding is None:
    df_model_binding = pd.DataFrame()
df_phase2_committed = globals().get("df_phase2_committed")
if df_phase2_committed is None:
    df_phase2_committed = pd.DataFrame()
df_rolling_fan = globals().get("df_rolling_fan")
if df_rolling_fan is None:
    df_rolling_fan = pd.DataFrame()

names_to_remove = ["Привязка_моделей", "Фаза2_committed", "Фаза2_rolling_fan"]
for rid, _, _, _, _ in sheet_triples:
    names_to_remove.extend(
        [f"{rid} Итог", f"{rid} Данные теста", f"{rid} По операциям", f"{rid} Предупреждения"]
    )

if out_path.exists():
    wb = load_workbook(out_path)
    for sn in names_to_remove:
        if sn in wb.sheetnames:
            wb.remove(wb[sn])
    if len(wb.sheetnames) == 0:
        wb.create_sheet("_tmp")
    wb.save(out_path)
    wb.close()
    writer_mode = "a"
else:
    writer_mode = "w"

with pd.ExcelWriter(out_path, engine="openpyxl", mode=writer_mode) as w:
    for rid, df_itog, df_detail, df_metrics_op, df_warn in sheet_triples:
        df_itog.to_excel(w, sheet_name=f"{rid} Итог", index=False)
        df_detail.to_excel(w, sheet_name=f"{rid} Данные теста", index=False)
        df_metrics_op.to_excel(w, sheet_name=f"{rid} По операциям", index=False)
        df_warn.to_excel(w, sheet_name=f"{rid} Предупреждения", index=False)
    if not df_model_binding.empty:
        df_model_binding.to_excel(w, sheet_name="Привязка_моделей", index=False)
    if not df_phase2_committed.empty:
        df_phase2_committed.to_excel(w, sheet_name="Фаза2_committed", index=False)
    if not df_rolling_fan.empty:
        df_rolling_fan.to_excel(w, sheet_name="Фаза2_rolling_fan", index=False)

wb = load_workbook(out_path)
if "_tmp" in wb.sheetnames and len(wb.sheetnames) > 1:
    wb.remove(wb["_tmp"])
    wb.save(out_path)
wb.close()

print(f"Excel: {out_path.resolve()}")
for rid, df_itog, _, _, df_warn in sheet_triples:
    print(
        f"  «{rid} Итог» ({len(df_itog)} строк), «{rid} Данные теста», «{rid} По операциям», "
        f"«{rid} Предупреждения» ({len(df_warn)} строк)"
    )


#### Выводы

* Вариант ноутбука с **несколькими** `RESIST_IDS` за один прогон; полный перебор комбинаций на каждый id.
* В сводке в ноутбуке — **топ-5** по «Качеству» **на каждый** `resist_id`; полная таблица — `df_results_full`.
* Excel (`OUT_XLSX`): на каждый id — четыре листа (**Итог**, **Данные теста**, **По операциям**, **Предупреждения**); плюс **Привязка_моделей**, **Фаза2_committed**, **Фаза2_rolling_fan** после ячейки «Сохранение моделей и фаза 2».
* Табличные лучшие модели — в `MODELS_DIR` (**joblib**); per-series на фазе 2 переобучаются без pickle.
